# Assignment 3 - Text Classification
The [Common Vulnerabilities and Exposures (CVE) framework](https://www.cve.org/) lists public vulnerabilities in software.
These CVEs are specific to a vendor and often even version of the software that is vulnerable, making it easy for security practitioner to identify if they are running vulnerable code.
However, due to the specificity of CVEs, it is difficult to know what kind of weakness the software exhibits.
Therefore, the [Common Weakness Enumeration (CWE) framework](https://cwe.mitre.org/) lists common weaknesses that perform an abstraction of CVEs.
While mapping a CVE to a CWE allows security operators to better understand how a vulnerability will impact their code, this mapping is often made manually.
In this assignment, you are given a dataset of CVE descriptions and their corresponding CWE entries and are tasked with creating a [DistilBERT](https://huggingface.co/docs/transformers/model_doc/distilbert)-based classifier that learns to automatically map CVEs to CWEs.

## Assignment submission
To submit this assignment, you will have to
- Submit a prediction for the `test` dataset to the https://security.eemcs.utwente.nl server.
- Upload the final version of your notebook (including the outputs) to Canvas.

### Server submission
On the https://security.eemcs.utwente.nl server, you will need to register an account if you haven't done so already. If you run into issues with registering an account, please contact the teacher of the course.

Navigate to `Challenges` -> `Cyber Data Analytics` -> `Assignment 3 - Text Classification`. Here you can start the docker conainer that will spin up the server that is required to answer question 10 of this assignment.

**Note:** If your performance is sufficient, you will receive a flag. Finding and/or submitting the flag is not a requirement for submitting and/or passing the assignment.

## Libraries
You will need the following Python libraries for this assignment:
- [pandas](https://pandas.pydata.org/)
- [pytorch](https://pytorch.org/)
- [scikit-learn](https://scikit-learn.org/stable/index.html)
- [transformers](https://huggingface.co/docs/transformers/installation)

## Load the data
**Question 1. [0.5pts]** Load the train and test data from `data/cve_train.csv` and `data/cve_test.csv` into a pandas dataframe.

In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv('./data/cve_train.csv')
test_df = pd.read_csv('./data/cve_test.csv')

In [3]:
train_df.head()

,identifier,description,cwe
0,CVE-2021-28969,eMPS 9.0.1.923211 on FireEye EX 3500 devices a...,CWE-89
1,CVE-2021-23862,A crafted configuration packet sent by an auth...,CWE-78
2,CVE-2023-21138,In onNullBinding of CallRedirectionProcessor.j...,CWE-20
3,CVE-2021-33732,A vulnerability has been identified in SINEC N...,CWE-89
4,CVE-2023-27592,"Miniflux is a feed reader. Since v2.0.25, Mini...",CWE-79


In [4]:
train_df['cwe'].unique()

<StringArray>
[ 'CWE-89',  'CWE-78',  'CWE-20',  'CWE-79', 'CWE-787', 'CWE-120', 'CWE-434',
 'CWE-125', 'CWE-862',  'CWE-22', 'CWE-352', 'CWE-416']
Length: 12, dtype: str

**Question 2. [0.5pts]** The train dataset contains labels of 12 different CWEs. For each CWE, find the corresponding name in the [CWE database](https://cwe.mitre.org/).

## Preprocessing
The target CWEs all have names in natural text. As we know, when classifying using a neural network, each output node represents a class. Therefore, we need to map the CWEs to indexes representing the output nodes, in our case 12.

**Question 3. [0.5pts]** Map the target CWEs to their corresponding output index.

In [7]:
unique_cwes = sorted(train_df['cwe'].unique())   # should be 12
cwe_to_idx = {cwe: i for i, cwe in enumerate(unique_cwes)}

train_df['label'] = train_df['cwe'].map(cwe_to_idx)

In [8]:
train_df.drop(['cwe'], axis=1, inplace=True)

## Train-validation split
Next, we will split the training input into a train and validation set that we can use to train and validate our network.

**Question 4. [0.5pts]** Split the training data into a train and validation set in an 80:20 ratio.

In [9]:
from sklearn.model_selection import train_test_split

train, validate = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'],
)

## Tokenizing
The first thing we need to do is transform the text data into tokens. In this assignment, we will use [DistilBERT](https://huggingface.co/docs/transformers/model_doc/distilbert) to perform our classification. Therefore, we must load the corresponding [DistilBERT tokenizer](https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertTokenizer) and tokenize the text.

**Question 5. [1.5pts]** Implement `TextDataset` to create a dataset of tokenized text from the dataframe using the `'distilbert-base-cased'` tokenizer. Ensure you add special tokens and perform both padding and truncation. You can assume a maximum length of 512 tokens. Use this `TextDataset` to create tokenized versions of the `train`, `validate` and `test` text data.

*Note: The test dataset does not contain target values. In this case, you can simply return -1 as a target value.*

In [10]:
from transformers import AutoTokenizer
import torch

/home/paul/Projects/University/cyber_data_analytics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from torch.utils.data import Dataset

class TextDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe
        self.tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-cased")

    def __getitem__(self, index):
        row = self.data.iloc[index]
        text = row['description']
        inputs = self.tokenizer(text, return_tensors="pt", padding="max_length", truncation=True)
        if 'label' in row:
            target = row['label']
        else:
            target = -1

        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'targets': target,
        }
    
    def __len__(self):
        return len(self.data)

In [12]:
train = TextDataset(train)
validate = TextDataset(validate)
test = TextDataset(test_df)

## Creating the classifier
We create our classifier based on the [DistilBERT](https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertModel) model. To this end, we use a pre-trained DistilBERT as a base and fine-tune our model for the downstream task of classification.

**Question 6. [1pts]** Implement `CveClassifier` using the pretrained `'distilbert-base-cased'` model as a first layer, a second dense layer with ReLU activation and `0.3` dropout, and a final classification layer, including `Softmax` function.

*Hint: Remember that the first (`0`) index of the output layer of a BERT model is used as input for a classification task.*

In [ ]:
# Imports
import torch.nn as nn
from transformers import DistilBertModel

class CveClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-cased")
        self.layer_1 = nn.Linear(768, 256)
        self.activation_1 = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, 12)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.layer_1(cls_output)
        x = self.activation_1(x)
        x = self.dropout(x)
        x = self.classifier(x)
        x = self.softmax(x)

        return x
        

## Training
Now that we have created our classifier, we can use our training data to train the classifier.

**Question 7. [2.5pts]** Implement `train_single_epoch` and train your network for a single epoch.

*Note: Transformer-based networks are often very large, meaning that training can take quite some time. Hence, before performing a full epoch of training, we recommend to test your implementation using a smaller dataset.*

In [ ]:
# Imports
from tqdm import tqdm
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# Select device on which to run
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CveClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

# Define how we train during an epoch
def train_single_epoch(network: nn.Module, data: Dataset) -> None: # I changed this to None, I don't see the point in returning a nn.Module here
    network.train()

    dataloader = DataLoader(data, batch_size=8, shuffle=True)

    total_loss = 0.0

    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].long().to(device)

        optimizer.zero_grad()

        outputs = network(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)
    print(f'Average training loss: {average_loss:.4f}')

train_single_epoch(model, train)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 31898.27it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 2886/2886 [08:02<00:00,  5.98it/s]

Average training loss: 1.8070


## Saving/Loading
As training takes a **very** long time, save your model to disk, so that you can re-use it later!

**Question 8a. [0.5pts]** Save your model to disk.

In [16]:
torch.save(model.state_dict(), "cve_classifier.pt")

**Question 8b. [0.5pts]** Load your model from disk.

In [17]:
model = CveClassifier().to(device)
model.load_state_dict(torch.load("cve_classifier.pt", map_location=device))
model.eval()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 30312.24it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CveClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
      

## Evaluating
Now that we have trained a neural network, we can make a prediction on our validation data. This allows us to measure the expected performance of the model.

**Question 9. [1pts]** Implement `evaluate` that returns the `y_true` (if any) and `y_pred` values., Predict the validation data and measure the performance using a `classification_report`.

In [18]:
# Imports
from sklearn.metrics import classification_report

def evaluate(network: nn.Module, data: Dataset) -> tuple[torch.Tensor, torch.Tensor]:
    network.eval()

    dataloader = DataLoader(
        data,
        batch_size=8,
        shuffle=False,
    )

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['targets'].long().to(device)

            outputs = network(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            predictions = torch.argmax(outputs, dim=1)

            y_true.append(targets.cpu())
            y_pred.append(predictions.cpu())

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)

    return y_true, y_pred


# Perform evaluation
y_true, y_pred = evaluate(model, validate)

# Print performance
print(classification_report(
    y_true = y_true.detach().cpu().numpy(),
    y_pred = y_pred.detach().cpu().numpy(),
    digits = 4,
))

100%|██████████| 722/722 [00:38<00:00, 18.84it/s]

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       191
           1     0.9022    0.9121    0.9071       364
           2     0.7409    0.6745    0.7062       212
           3     0.9135    0.9076    0.9105       314
           4     0.9812    0.9812    0.9812       373
           5     0.9504    0.9404    0.9453       285
           6     0.9021    0.8413    0.8706       208
           7     0.9109    0.8929    0.9018       252
           8     0.7827    0.9348    0.8521       844
           9     0.9608    0.9925    0.9764      1729
          10     0.8566    0.8972    0.8764       253
          11     0.9919    0.9880    0.9900       748

    accuracy                         0.9120      5773
   macro avg     0.8244    0.8302    0.8265      5773
weighted avg     0.8846    0.9120    0.8971      5773




/home/paul/Projects/University/cyber_data_analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/paul/Projects/University/cyber_data_analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/paul/Projects/University/cyber_data_analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete

## Optimize (optional)
Optimize your model to achieve a better performance. You can try to change e.g.,:
 - the base model;
 - the number of additional layers in your classifier;
 - different training data/method.

## Prediction
Now that you have trained (and possibly optimized) your classifier, we will perform a final prediction on the `test` data. This prediction is validated by the server using the `submit` method.

Below you will find the requirements for gaining the additional points for Question 10.

| Requirement                                      | Reward     |
|--------------------------------------------------|------------|
| Weighted F1 score >= 0.7                         | 0.5 points |
| Macro F1 score >= 0.7                            | 0.5 points |
| Weighted F1 score >= 0.5 & Macro F1 score >= 0.5 | Flag       |

**Question 10. [1pts]** Predict the `test` data and send your submission to the server.

*Note: The server is rate-limited so it may take a few seconds before you receive a reply.*

In [19]:
# Perform evaluation
y_true, y_pred = evaluate(model, test)

100%|██████████| 902/902 [00:45<00:00, 19.78it/s]


In [24]:
import requests

def submit(url: str, y_pred: list[int], mapping: dict[str, int]) -> None:
    print(requests.post(url, json={'y_pred': y_pred.tolist(), 'mapping': mapping}).text)

submit(
    url = 'http://security.eemcs.utwente.nl:33138', # Change this with the URL from security.eemcs.utwente.nl
    y_pred = y_pred,   # Your prediction
    mapping = cwe_to_idx, # Your mapping from CWE to index in the form dict[str, int] 
)

              precision    recall  f1-score   support

     CWE-120     0.0000    0.0000    0.0000       263
     CWE-125     0.9023    0.9044    0.9034       429
      CWE-20     0.7598    0.6541    0.7030       266
      CWE-22     0.9127    0.9200    0.9163       375
     CWE-352     0.9752    0.9752    0.9752       483
     CWE-416     0.9231    0.9385    0.9307       358
     CWE-434     0.9095    0.8277    0.8667       267
      CWE-78     0.9049    0.8990    0.9020       307
     CWE-787     0.7498    0.9275    0.8292      1021
      CWE-79     0.9568    0.9858    0.9711      2178
     CWE-862     0.8647    0.9046    0.8842       325
      CWE-89     0.9915    0.9841    0.9878       943

    accuracy                         0.9046      7215
   macro avg     0.8208    0.8268    0.8225      7215
weighted avg     0.8758    0.9046    0.8886      7215


Flag = THS{Tr4nsf0rm3rs_fr0m_s3Sam3_Str33t}


### To submit the assignment, please upload the final version of your notebook to Canvas!